In [2]:
# ============================================================================
# 07_quality_and_fidelity.ipynb
# LLM experiment, part 2 (requires an OpenAI key in ../.env). Three questions:
#   (A) Fidelity: does the generator report the evidence it was handed, and
#       does a constrained prompt remove the omission gap? (per condition)
#   (B) Evaluation blind spot (RQ5): are LLM-as-a-judge quality scores related
#       to how much two explanations of the same case differ in content?
#   (C) Narrative coherence: a deliberately mis-scaled "overall verdict" is used
#       as a controlled natural experiment to test whether a generator rewrites
#       a top driver's direction to agree with a contradictory verdict, and
#       whether the constrained prompt eliminates it -- tested UNDER the
#       contradictory verdict, with a placebo stratum and interval estimates.
# Two judges from different model generations score each explanation in
# isolation. All calls are cached; no file path is printed.
# ============================================================================

# [Imports and global configuration]
import os
import re
import json
import hashlib
import warnings
from pathlib import Path
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

ROOT = Path("..")
DATA_DIR = ROOT / "data"
PROMPT_DIR = ROOT / "prompts"
GEN_DIR = DATA_DIR / "generations"
CLAIM_DIR = DATA_DIR / "claims_cache"
JUDGE_DIR = DATA_DIR / "judge_cache"
FIG_DIR = ROOT / "results" / "figures"
TAB_DIR = ROOT / "results" / "tables"
for d in (JUDGE_DIR, FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

_env = find_dotenv(usecwd=True)
if _env:
    load_dotenv(_env)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found; add it to ../.env before running.")
client = OpenAI()
print("credentials loaded")

# [Plotting style: grayscale, publication settings, no captions]
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette(sns.color_palette("gray", n_colors=8))
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "font.family": "sans-serif",
    "axes.grid": True, "grid.color": "0.85", "grid.linewidth": 0.5,
    "axes.edgecolor": "0.3", "axes.labelcolor": "0.15", "text.color": "0.15",
    "xtick.color": "0.3", "ytick.color": "0.3", "image.cmap": "gray",
    "savefig.bbox": "tight", "savefig.transparent": False,
})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"saved figure: {name}")

def save_table(df, name, index=False):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index, encoding="utf-8-sig")
    print(f"saved table: {name}")

# [Configuration]
GENERATORS = {"weak": os.getenv("LLM_MODEL_WEAK", "gpt-4o-mini"),
              "reference": os.getenv("LLM_MODEL_MID", "gpt-5.4-mini"),
              "strong": os.getenv("LLM_MODEL_STRONG", "gpt-5.4")}
# Two judges from different generations; scores averaged to mitigate self-preference.
JUDGES = {"judge_a": os.getenv("LLM_MODEL_MID", "gpt-5.4-mini"),
          "judge_b": os.getenv("LLM_MODEL_WEAK", "gpt-4o-mini")}
REFERENCE_GENERATOR = "reference"
REFERENCE_MODEL, REFERENCE_XAI, REFERENCE_PROMPT = "XGBoost", "SHAP", "basic"
QUALITY_HIGH = 4.0
MAX_JUDGE_PER_CELL = None   # cap explanations judged per condition (None = all)
B_BOOTSTRAP = 2000
TOP_K = 5

with open(DATA_DIR / "feature_registry.json", encoding="utf-8") as f:
    registry = json.load(f)
corpus = pd.read_parquet(DATA_DIR / "claims_corpus.parquet")
evidence = pd.read_json(DATA_DIR / "evidence_blocks.json")
evidence["key"] = (evidence["domain"] + "|" + evidence["case_id"].astype(str) + "|"
                   + evidence["model"] + "|" + evidence["xai"])
ev_lookup = {k: d for k, d in zip(evidence["key"], evidence["drivers"])}
ev_meta = {k: (bv, pp) for k, bv, pp in
           zip(evidence["key"], evidence["base_value"], evidence["predicted_probability"])}
# Case-level adverse-decision flag (the "rescue target" subgroup), carried from
# notebook 03 so fidelity and the blind-spot can be reported for the full sample
# and for the adverse subgroup.
adverse_by_case = {(d, str(c)): bool(a) for d, c, a in
                   zip(evidence["domain"], evidence["case_id"], evidence.get("adverse", False))}
adverse_applicable = {d: bool(x) for d, x in
                      zip(evidence["domain"], evidence.get("adverse_applicable", False))}
print(f"corpus: {len(corpus):,} explanations")

def cache_key(*p):
    return hashlib.sha1("||".join(map(str, p)).encode()).hexdigest()

def chat(model, system, user, cache_dir, max_tokens=300, seed=SEED):
    key = cache_key(model, system, user, seed, max_tokens)
    fp = cache_dir / f"{key}.json"
    if fp.exists():
        return fp.read_text(encoding="utf-8")
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    kwargs = dict(model=model, messages=msgs, max_completion_tokens=max_tokens, seed=seed)
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(**kwargs)
            text = resp.choices[0].message.content or ""
            fp.write_text(text, encoding="utf-8")
            return text
        except TypeError:
            kwargs.pop("seed", None); kwargs["max_tokens"] = kwargs.pop("max_completion_tokens", max_tokens)
        except Exception:
            if attempt == 2:
                raise
    return ""

# [A) Fidelity of each explanation to the evidence it was given]
def eval_fidelity(row):
    key = f"{row['domain']}|{row['case_id']}|{row['model']}|{row['xai']}"
    drivers = ev_lookup.get(key)
    if not drivers:
        return None
    ev_feats = {d["feature"] for d in drivers[:TOP_K]}
    ev_dir = {d["feature"]: d["direction"] for d in drivers[:TOP_K]}
    cl = [c for c in row["claims"] if isinstance(c, dict) and "feature" in c]
    cl_feats = {c["feature"] for c in cl}
    cl_dir = {c["feature"]: c.get("direction") for c in cl}
    omission = 1 - (len(ev_feats & cl_feats) / len(ev_feats) if ev_feats else 1.0)
    hallucination = len(cl_feats - ev_feats) / len(cl_feats) if cl_feats else 0.0
    shared = ev_feats & cl_feats
    dir_fid = np.mean([ev_dir[f] == cl_dir[f] for f in shared]) if shared else np.nan
    return pd.Series({"omission": omission, "hallucination": hallucination,
                      "direction_fidelity": dir_fid})

fid = corpus.join(corpus.apply(eval_fidelity, axis=1))
fid["adverse"] = [adverse_by_case.get((d, str(c)), False)
                  for d, c in zip(fid["domain"], fid["case_id"])]
block_b = fid[(fid.model == REFERENCE_MODEL) & (fid.xai == REFERENCE_XAI) & (fid.rep == 0)]
fid_cond = (block_b.groupby(["generator", "prompt"])[["omission", "hallucination", "direction_fidelity"]]
            .mean().round(4).reset_index())
save_table(fid_cond, "tab33_fidelity_by_condition")
print(fid_cond.to_string(index=False))
# Same fidelity breakdown restricted to the adverse-decision subgroup.
adv_b = block_b[block_b.adverse]
if len(adv_b):
    fid_cond_adv = (adv_b.groupby(["generator", "prompt"])[["omission", "hallucination", "direction_fidelity"]]
                    .mean().round(4).reset_index())
    save_table(fid_cond_adv, "tab33b_fidelity_by_condition_adverse")
    print("adverse subgroup fidelity:\n", fid_cond_adv.to_string(index=False))

# [Figure 25: fidelity by prompt (omission falls to zero under the constraint)]
fig, ax = plt.subplots(figsize=(6.6, 3.6))
pv = block_b.groupby("prompt")[["omission", "hallucination"]].mean()
x = np.arange(len(pv)); w = 0.38
ax.bar(x - w / 2, pv["omission"], width=w, color="0.5", edgecolor="0.15", label="omission")
ax.bar(x + w / 2, pv["hallucination"], width=w, color="0.25", edgecolor="0.15", label="hallucination")
ax.set_xticks(x); ax.set_xticklabels(pv.index, fontsize=8)
ax.set_ylabel("Rate"); ax.legend(fontsize=7, frameon=False)
save_fig(fig, "fig25_fidelity_by_condition")

# [B) Judge every explanation in isolation, then test the blind spot]
JUDGE_SYSTEM = (
    "You are grading one explanation of a model's assessment, on its own. "
    "Score four dimensions from 1 (poor) to 5 (excellent): accuracy (consistency "
    "with the listed evidence, including the direction of each factor), clarity, "
    "usefulness to a non-expert, and completeness. Return JSON only: "
    '{"accuracy": n, "clarity": n, "usefulness": n, "completeness": n}.')

def judge_user(row):
    key = f"{row['domain']}|{row['case_id']}|{row['model']}|{row['xai']}"
    drivers = ev_lookup.get(key, [])
    ev_lines = "\n".join(f"  {d['rank']}. {d['feature'].replace('_',' ')} "
                         f"({'up' if d['direction']=='increases' else 'down'})"
                         for d in drivers[:TOP_K])
    return f"EVIDENCE\n{ev_lines}\n\nEXPLANATION\n{row['explanation']}"

def parse_scores(raw):
    try:
        d = json.loads(re.sub(r"^```[a-z]*|```$", "", raw.strip(), flags=re.M))
        return [float(d[k]) for k in ("accuracy", "clarity", "usefulness", "completeness")]
    except Exception:
        return [np.nan] * 4

# Judge the explanations used in the pair analyses (evidence- and generation-stage
# reference conditions), optionally capped per condition.
judge_pool = fid[fid.rep == 0].copy()
if MAX_JUDGE_PER_CELL is not None:
    judge_pool = (judge_pool.groupby(["domain", "generator", "prompt", "xai", "model"], group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), MAX_JUDGE_PER_CELL), random_state=SEED)))

def run_judge(i):
    row = judge_pool.loc[i]
    user = judge_user(row)
    out = {}
    for jname, jmodel in JUDGES.items():
        acc, cla, use, comp = parse_scores(chat(jmodel, JUDGE_SYSTEM, user, JUDGE_DIR, max_tokens=120))
        out[jname] = (acc, cla, use, comp)
    return i, out

scores = {}
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(run_judge, i) for i in judge_pool.index]
    done = 0
    for fut in as_completed(futs):
        i, out = fut.result(); scores[i] = out; done += 1
        if done % 500 == 0:
            print(f"judged {done:,}/{len(judge_pool):,}")

def mean_quality(i):
    vals = scores.get(i)
    if not vals:
        return np.nan, np.nan
    arrs = np.array([vals[j] for j in JUDGES])
    overall = np.nanmean(arrs)
    accuracy = np.nanmean(arrs[:, 0])
    return overall, accuracy
judge_pool["quality"] = [mean_quality(i)[0] for i in judge_pool.index]
judge_pool["judge_accuracy"] = [mean_quality(i)[1] for i in judge_pool.index]
# Inter-judge agreement (Spearman over the overall score of each judge).
qa = np.array([np.nanmean(scores[i]["judge_a"]) for i in judge_pool.index])
qb = np.array([np.nanmean(scores[i]["judge_b"]) for i in judge_pool.index])
mask = ~(np.isnan(qa) | np.isnan(qb))
inter_judge = float(spearmanr(qa[mask], qb[mask]).correlation) if mask.sum() > 2 else np.nan
save_table(pd.DataFrame([{"inter_judge_spearman": round(inter_judge, 3),
                          "n_judged": int(len(judge_pool))}]), "tab54_judge_agreement")

# [Build same-case pairs that differ in exactly one factor]
def claims_feats(cl):
    return frozenset(c["feature"] for c in cl if isinstance(c, dict) and "feature" in c)
qlook = {i: judge_pool.loc[i, "quality"] for i in judge_pool.index}
clook = {}
for i in judge_pool.index:
    r = judge_pool.loc[i]
    clook[(r["domain"], r["case_id"], r["model"], r["xai"], r["generator"], r["prompt"])] = i

def make_pairs(vary):
    rows = []
    for domain in registry:
        for cid in corpus[corpus.domain == domain]["case_id"].unique():
            if vary == "xai":
                a = clook.get((domain, cid, REFERENCE_MODEL, "SHAP", REFERENCE_GENERATOR, REFERENCE_PROMPT))
                b = clook.get((domain, cid, REFERENCE_MODEL, "LIME", REFERENCE_GENERATOR, REFERENCE_PROMPT))
            else:
                a = clook.get((domain, cid, REFERENCE_MODEL, REFERENCE_XAI, "reference", REFERENCE_PROMPT))
                b = clook.get((domain, cid, REFERENCE_MODEL, REFERENCE_XAI, "strong", REFERENCE_PROMPT))
            if a is None or b is None:
                continue
            fa, fb = claims_feats(judge_pool.loc[a, "claims"]), claims_feats(judge_pool.loc[b, "claims"])
            union = fa | fb
            feat_dist = 1 - (len(fa & fb) / len(union) if union else 1.0)
            qa_, qb_ = qlook[a], qlook[b]
            rows.append({"domain": domain, "case_id": cid, "vary": vary,
                         "feature_distance": feat_dist, "quality_gap": abs(qa_ - qb_),
                         "both_high": (qa_ >= QUALITY_HIGH and qb_ >= QUALITY_HIGH),
                         "different_drivers": feat_dist > 0})
    return pd.DataFrame(rows)

pairs = pd.concat([make_pairs("xai"), make_pairs("generator")], ignore_index=True)
pairs.to_parquet(DATA_DIR / "quality_pairs.parquet", index=False)
m = pairs[["feature_distance", "quality_gap"]].dropna()
rho = float(spearmanr(m["feature_distance"], m["quality_gap"]).correlation) if len(m) > 2 else np.nan
blind = pairs[(pairs.vary == "xai") & (pairs.both_high)]
blind_rate = float(blind["different_drivers"].mean()) if len(blind) else np.nan
uncond = float(pairs[pairs.vary == "xai"]["different_drivers"].mean())
save_table(pd.DataFrame([{"quality_distance_spearman": round(rho, 3),
                          "blind_spot_both_high_diff_drivers": round(blind_rate, 3),
                          "unconditional_diff_drivers": round(uncond, 3),
                          "inter_judge_spearman": round(inter_judge, 3)}]),
           "tab56_quality_distance_correlation")
print(f"quality~distance rho={rho:.3f} | blind spot={blind_rate:.3f} "
      f"| unconditional={uncond:.3f} | inter-judge={inter_judge:.3f}")

# [Figure 36: quality gap vs content distance]
fig, ax = plt.subplots(figsize=(5.4, 4.0))
ax.scatter(m["feature_distance"], m["quality_gap"], s=8, color="0.45", alpha=0.5, edgecolor="none")
ax.set_xlabel("Content distance (feature)"); ax.set_ylabel("Quality-score gap")
ax.text(0.03, 0.95, f"Spearman rho = {rho:.3f}", transform=ax.transAxes, fontsize=8, va="top")
save_fig(fig, "fig36_quality_versus_distance")

# [Figure 37: the evaluation blind spot]
fig, ax = plt.subplots(figsize=(5.0, 3.6))
cats = ["XAI method", "Generator"]
vals = [blind_rate,
        float(pairs[(pairs.vary == "generator") & (pairs.both_high)]["different_drivers"].mean())]
ax.bar(range(2), vals, color=["0.3", "0.6"], edgecolor="0.15", width=0.55)
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f"{v:.1%}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(range(2)); ax.set_xticklabels(cats, fontsize=9)
ax.set_ylabel("Both score >=4 yet name different drivers"); ax.set_ylim(0, 1.05)
save_fig(fig, "fig37_quality_blind_spot")

# [Figure 39: judge accuracy vs mechanically measured direction fidelity]
jf = judge_pool.dropna(subset=["judge_accuracy", "direction_fidelity"])
rho_acc = float(spearmanr(jf["judge_accuracy"], jf["direction_fidelity"]).correlation) if len(jf) > 2 else np.nan
rho_om = float(spearmanr(jf["judge_accuracy"], jf["omission"]).correlation) if len(jf) > 2 else np.nan
fig, ax = plt.subplots(figsize=(5.2, 3.8))
ax.scatter(jf["direction_fidelity"], jf["judge_accuracy"], s=8, color="0.4", alpha=0.5, edgecolor="none")
ax.set_xlabel("Measured direction fidelity"); ax.set_ylabel("Judge accuracy score")
ax.text(0.03, 0.05, f"rho(dir)={rho_acc:.2f}  rho(omission)={rho_om:.2f}",
        transform=ax.transAxes, fontsize=8)
save_fig(fig, "fig39_judge_accuracy_versus_fidelity")

# [C) Controlled natural experiment on narrative coherence]
# For cases whose TOP driver opposes the overall verdict, regenerate the
# reference explanation under a deliberately mis-scaled verdict (every case
# presented as "above the typical level"). The placebo stratum holds cases
# whose real verdict is already "above". Reversal = the generator states the
# top driver's direction opposite to the supplied evidence.
PROMPTS = {name: (PROMPT_DIR / f"{name}_v1.txt").read_text(encoding="utf-8")
           for name in ("basic", "constrained")}

def evidence_text_scaled(drivers, base_value, pred_prob, force_above=False):
    verdict = "above" if (force_above or pred_prob >= base_value) else "below"
    lines = [f"Overall, this case scores {verdict} the typical level.", "Factors, most important first:"]
    for d in drivers[:TOP_K]:
        updown = "raised" if d["direction"] == "increases" else "lowered"
        lines.append(f"  {d['rank']}. {d['feature'].replace('_',' ')} -- {updown} the assessment")
    return "\n".join(lines)

def top_dir_stated(claims, top_feature):
    for c in claims:
        if isinstance(c, dict) and c.get("feature") == top_feature:
            return c.get("direction")
    return None

# Reference-condition cases of the primary domain.
prim = next(d for d in registry if registry[d]["primary"])
ref = corpus[(corpus.domain == prim) & (corpus.model == REFERENCE_MODEL) &
             (corpus.xai == REFERENCE_XAI) & (corpus.generator == REFERENCE_GENERATOR) &
             (corpus.prompt == REFERENCE_PROMPT) & (corpus.rep == 0)]
natexp_rows = []
def gen_scaled(domain, cid, gen_tier, prompt_name, force_above):
    key = f"{domain}|{cid}|{REFERENCE_MODEL}|{REFERENCE_XAI}"
    drivers = ev_lookup[key]; bv, pp = ev_meta[key]
    user = f"EVIDENCE\n{evidence_text_scaled(drivers, bv, pp, force_above)}\n\nWrite the explanation now."
    raw = chat(GENERATORS[gen_tier], PROMPTS[prompt_name], user, GEN_DIR, max_tokens=400,
               seed=SEED + (7 if force_above else 0))
    return raw

with open(DATA_DIR / "extractor_config.json", encoding="utf-8") as f:
    EXTRACTOR = json.load(f)
def extract(domain, text):
    allowed = ", ".join(registry[domain]["features"])
    raw = chat(EXTRACTOR["extractor_model"], EXTRACTOR["extractor_system_prompt"],
               f"ALLOWED FEATURES\n{allowed}\n\nEXPLANATION\n{text}", CLAIM_DIR, max_tokens=700)
    try:
        data = json.loads(re.sub(r"^```[a-z]*|```$", "", raw.strip(), flags=re.M))
    except Exception:
        return []
    # The model may return a bare list, or wrap the list in a dict; keep only
    # well-formed claim objects so downstream indexing is always safe.
    if isinstance(data, dict):
        data = data.get("claims", data.get("features", []))
    return [c for c in data if isinstance(c, dict) and "feature" in c]

# Build the task grid, then run the generation + extraction in parallel with a
# progress counter (the same pattern as the generation and judging phases), so
# this stage reports progress instead of appearing to hang.
natexp_tasks = []
for _, r in ref.iterrows():
    key = f"{prim}|{r['case_id']}|{REFERENCE_MODEL}|{REFERENCE_XAI}"
    drivers = ev_lookup.get(key)
    if not drivers:
        continue
    bv, pp = ev_meta[key]
    top = drivers[0]
    opposes = (top["direction"] == "decreases" and pp >= bv) or (top["direction"] == "increases" and pp < bv)
    stratum = "placebo" if pp >= bv else "treatment"   # treatment = real verdict below, mis-scaled to above
    for gen_tier in GENERATORS:
        for prompt_name in ("basic", "constrained"):
            for run, force in (("correct", False), ("misscaled", True)):
                natexp_tasks.append((r["case_id"], top, opposes, stratum, gen_tier, prompt_name, run, force))

def run_natexp(task):
    cid, top, opposes, stratum, gen_tier, prompt_name, run, force = task
    text = gen_scaled(prim, cid, gen_tier, prompt_name, force)
    cl = extract(prim, text)
    stated = top_dir_stated(cl, top["feature"])
    reversed_ = int(stated is not None and stated != top["direction"])
    return {"case_id": cid, "generator": gen_tier, "prompt": prompt_name, "run": run,
            "stratum": stratum, "top_opposes_verdict": opposes, "reversed": reversed_}

print(f"natural experiment: {len(natexp_tasks):,} generations to run")
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(run_natexp, t) for t in natexp_tasks]
    done = 0
    for fut in as_completed(futs):
        natexp_rows.append(fut.result()); done += 1
        if done % 100 == 0 or done == len(natexp_tasks):
            print(f"natural experiment {done:,}/{len(natexp_tasks):,}")
nat = pd.DataFrame(natexp_rows)
nat.to_parquet(DATA_DIR / "natural_experiment.parquet", index=False)

# Difference-in-differences of the top-driver reversal rate (basic prompt).
def did_table(prompt_name):
    sub = nat[nat.prompt == prompt_name]
    rows = []
    for gen_tier in GENERATORS:
        g = sub[sub.generator == gen_tier]
        def rate(stratum, run):
            x = g[(g.stratum == stratum) & (g.run == run)]["reversed"]
            return float(x.mean()) if len(x) else np.nan
        t = rate("treatment", "misscaled") - rate("treatment", "correct")
        p = rate("placebo", "misscaled") - rate("placebo", "correct")
        rows.append({"generator": gen_tier,
                     "treatment_misscaled": round(rate("treatment", "misscaled"), 4),
                     "treatment_correct": round(rate("treatment", "correct"), 4),
                     "placebo_misscaled": round(rate("placebo", "misscaled"), 4),
                     "did": round((t - p), 4) if (t == t and p == p) else np.nan})
    return pd.DataFrame(rows)
did_basic = did_table("basic"); did_constrained = did_table("constrained")
save_table(did_basic, "tab72_reversal_did_basic")
save_table(did_constrained, "tab72_reversal_did_constrained")
print("natural experiment (basic prompt):\n", did_basic.to_string(index=False))
print("under constrained prompt:\n", did_constrained.to_string(index=False))

# [Figure 41/42: reversal DiD by generator, and effect of the constrained prompt]
fig, ax = plt.subplots(figsize=(6.2, 3.6))
x = np.arange(len(GENERATORS)); w = 0.38
ax.bar(x - w / 2, did_basic["did"], width=w, color="0.35", edgecolor="0.15", label="basic prompt")
ax.bar(x + w / 2, did_constrained["did"], width=w, color="0.65", edgecolor="0.15", label="constrained prompt")
ax.axhline(0, color="0.2", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(list(GENERATORS.keys()), fontsize=8)
ax.set_ylabel("Reversal rate difference-in-differences")
ax.legend(fontsize=7, frameon=False)
save_fig(fig, "fig42_reversal_mechanism")

print("\nQuality + fidelity analysis complete.")

credentials loaded
corpus: 28,000 explanations
saved table: tab33_fidelity_by_condition
generator       prompt  omission  hallucination  direction_fidelity
reference        basic    0.0950         0.0203              0.9943
reference  constrained    0.0121         0.0000              1.0000
reference verification    0.0347         0.0024              1.0000
   strong        basic    0.0182         0.0013              0.9985
   strong  constrained    0.0069         0.0000              1.0000
   strong verification    0.0142         0.0002              1.0000
     weak        basic    0.1818         0.0225              0.9469
     weak  constrained    0.0486         0.0056              0.9985
     weak verification    0.1445         0.0165              0.9825
saved table: tab33b_fidelity_by_condition_adverse
adverse subgroup fidelity:
 generator       prompt  omission  hallucination  direction_fidelity
reference        basic    0.1188         0.0327              0.9912
reference  constra